# Model Interpretation: Real / Fake, Gender and Age Group (with Grad-CAM)

Give **one picture** and this notebook shows:

1. **Real or fake?** (with confidence and a Grad-CAM heatmap)
2. **Which gender?** (only for images judged real; with Grad-CAM)
3. **Which age group?** (only for images judged real; with Grad-CAM)

For an image judged **fake**, gender is shown as *Unknown* and age as *Not applicable*. This is the same rule used for the fake images in the dataset.

Three separately trained models are used, one per task (see the settings cell). Grad-CAM shows **where a model looked** for its answer (red = strong influence). It does not prove the answer is correct.

### Import Libraries:

In [11]:
import io
import random
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pathlib import Path
from PIL import Image
from torchvision import transforms, models

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


### Settings:

Change only the values in this cell.

In [12]:
# --------------------------------------------------
# Change only these values
# --------------------------------------------------

PROJECT_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final")
MODELS_DIR = PROJECT_DIR / "models"
DATA_DIR = PROJECT_DIR / "data"
CHART_DIR = PROJECT_DIR / "charts"

# The picture to analyse: the full path of ONE image file.
# None -> a random image from the real/fake test set is used.
IMAGE_PATH = None            # example: r"C:\Users\jagad\Pictures\my_photo.jpg"

# If the confidence is below this value, the answer is shown as "Unknown".
GENDER_MIN_CONF = 0.60
AGE_MIN_CONF = 0.40          # lower, because age has more classes and is harder

SAVE_FIG = False             # True -> also save the figure as a .pkl file in the models folder
SEED = None                  # None -> different random images every run; a number -> the same ones

# --------------------------------------------------
# Which saved model is used for each task
#   arch    : cnn, resnet50, vgg16, efficientnet_b0 or vit_b16
#   classes : the class_names printed in that task's training notebook (same order)
#   prep    : "norm" = centre-crop to a square, 256 x 256, JPEG quality 95
#                      (how the real/fake and gender training images were made)
#             "raw"  = the image as it is
#                      (the age model was trained on the original split_age images)
#   exclude : classes that are never predicted for a real photo
# --------------------------------------------------
TASKS = {
    "real_fake": {
        "arch": "resnet50",
        "file": MODELS_DIR / "real_fake" / "resnet50_model_real_fake.pth",
        "classes": ["fake", "real"],
        "prep": "norm",
        "exclude": [],
    },
    "gender": {
        "arch": "vit_b16",
        "file": MODELS_DIR / "gender" / "vit_model_gender.pth",
        "classes": ["female", "male"],
        "prep": "norm",
        "exclude": [],
    },
    "age": {
        "arch": "resnet50",
        "file": MODELS_DIR / "age_group" / "resnet50_age_group.pth",
        "classes": ["18-25", "26-35", "36-50", "50+", "not_applicable", "under_18"],
        "prep": "raw",
        "exclude": ["not_applicable"],
    },
}

# Test folders used to pick random example images
TEST_DIRS = {
    "real_fake": DATA_DIR / "split_real_fake_norm" / "test",
    "gender": DATA_DIR / "split_gender_norm" / "test",
    "age": DATA_DIR / "split_age" / "test",
}

if SEED is not None:
    random.seed(SEED)

### Image preparation:

Each model gets the image prepared the same way as its training images. After that, every model uses the same resize to 224 x 224 and the same normalization as the validation and test images.

In [13]:
def prepare_norm(img):
    """Centre-crop to a square, resize to 256 x 256, save as JPEG quality 95 (as for the training images)."""
    img = img.convert("RGB")
    w, h = img.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    img = img.crop((left, top, left + s, top + s)).resize((256, 256), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=95)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def prepare_raw(img):
    """The image as it is."""
    return img.convert("RGB")


PREP = {"norm": prepare_norm, "raw": prepare_raw}

to_tensor = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

### Load the three models and prepare Grad-CAM:

The model structures are rebuilt exactly as in the training notebooks, then the saved weights are loaded.
Grad-CAM uses the last convolutional block of each CNN. The Vision Transformer (ViT) has no convolutions, so it uses the layer norm of its last encoder block, and its tokens are reshaped back into a 14 x 14 grid.

In [14]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))

        self.block1 = block(3, 32)
        self.block2 = block(32, 64)
        self.block3 = block(64, 128)
        self.block4 = block(128, 256)
        self.block5 = block(256, 512)
        self.dropout_conv = nn.Dropout(0.3)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(512, 256)
        self.dropout_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.dropout_conv(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.dropout_fc(x)
        return self.fc2(x)


def build_model(arch, n):
    if arch == "cnn":
        return CNN(n)
    if arch == "resnet50":
        m = models.resnet50(weights=None)
        m.fc = nn.Sequential(nn.Linear(m.fc.in_features, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, n))
        return m
    if arch == "vgg16":
        m = models.vgg16(weights=None)
        m.classifier = nn.Sequential(nn.Linear(25088, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, n))
        return m
    if arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, 256),
                                     nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, n))
        return m
    if arch == "vit_b16":
        m = models.vit_b_16(weights=None)
        m.heads = nn.Sequential(nn.Linear(m.heads[0].in_features, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, n))
        return m
    raise ValueError(f"Unknown architecture: {arch}")


def get_target_layers(model, arch):
    """The layer that Grad-CAM explains."""
    if arch == "cnn":
        return [model.block5[0]]
    if arch == "resnet50":
        return [model.layer4[-1]]
    if arch == "vgg16":
        return [model.features[-1]]
    if arch == "efficientnet_b0":
        return [model.features[-1]]
    if arch == "vit_b16":
        return [model.encoder.layers[-1].ln_1]
    raise ValueError(f"Unknown architecture: {arch}")


def vit_reshape_transform(tensor, height=14, width=14):
    """ViT tokens (1 class token + 14 x 14 patches) -> a feature map (channels, 14, 14)."""
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    return result.transpose(2, 3).transpose(1, 2)


LOADED = {}

for task, cfg in TASKS.items():
    if not cfg["file"].exists():
        raise FileNotFoundError(f"Model file for '{task}' not found: {cfg['file']}")

    model = build_model(cfg["arch"], len(cfg["classes"]))
    model.load_state_dict(torch.load(cfg["file"], map_location="cpu"))
    model = model.to(device).eval()

    cam = GradCAM(
        model=model,
        target_layers=get_target_layers(model, cfg["arch"]),
        reshape_transform=vit_reshape_transform if cfg["arch"] == "vit_b16" else None,
    )

    LOADED[task] = {**cfg, "model": model, "cam": cam}
    print(f"{task:10s} {cfg['arch']:16s} classes = {cfg['classes']}  -> loaded")

real_fake  resnet50         classes = ['fake', 'real']  -> loaded
gender     vit_b16          classes = ['female', 'male']  -> loaded
age        resnet50         classes = ['18-25', '26-35', '36-50', '50+', 'not_applicable', 'under_18']  -> loaded


### Predict and explain one image with one model:

For each task the model gives a probability for every class. Grad-CAM then explains the **predicted** class.
For the age model, the class `not_applicable` is never chosen for a real photo (it is mostly the label of fake images in the dataset), and the other probabilities are rescaled to add up to 1.

In [15]:
def analyse(task, pil_image):
    cfg = LOADED[task]

    prepared = PREP[cfg["prep"]](pil_image)
    x = to_tensor(prepared).unsqueeze(0).to(device)

    # Probabilities
    with torch.no_grad():
        probs = torch.softmax(cfg["model"](x), dim=1)[0].cpu()

    for name in cfg["exclude"]:
        probs[cfg["classes"].index(name)] = 0.0
    probs = probs / probs.sum()

    idx = int(probs.argmax())

    # Grad-CAM for the predicted class
    gray = cfg["cam"](input_tensor=x, targets=[ClassifierOutputTarget(idx)])[0]

    # The image exactly as the model saw it (undo the normalization)
    rgb = x[0].cpu().permute(1, 2, 0).numpy()
    rgb = np.clip(rgb * IMAGENET_STD + IMAGENET_MEAN, 0, 1).astype(np.float32)

    return {
        "task": task,
        "label": cfg["classes"][idx],
        "confidence": float(probs[idx]),
        "probs": {c: float(p) for c, p in zip(cfg["classes"], probs)},
        "original": (rgb * 255).astype(np.uint8),
        "gradcam": show_cam_on_image(rgb, gray, use_rgb=True),
    }

### Full prediction for one picture:

1. The real/fake model decides.
2. **Fake** -> gender *Unknown*, age *Not applicable*.
3. **Real** -> the gender model and the age model run, each with its own Grad-CAM.
4. If the confidence of gender or age is below the limit in the settings, that answer is shown as *Unknown*.

In [16]:
try:
    TRUTH = pd.read_csv(DATA_DIR / "final_image_dataset.csv", dtype=str).set_index("image_id")
except FileNotFoundError:
    TRUTH = None


def true_labels(path):
    """If the picture is one of the dataset images, return its known labels."""
    if TRUTH is not None and Path(path).stem in TRUTH.index:
        r = TRUTH.loc[Path(path).stem]
        return f"{r['label']} | gender: {r['gender']} | age group: {r['age_group']}"
    return None


def full_prediction(pil_image):
    results = {"real_fake": analyse("real_fake", pil_image)}
    rf = results["real_fake"]

    if rf["label"] == "fake":
        gender_text, age_text = "Unknown", "Not applicable"
    else:
        results["gender"] = analyse("gender", pil_image)
        results["age"] = analyse("age", pil_image)
        g, a = results["gender"], results["age"]
        gender_text = g["label"].capitalize() if g["confidence"] >= GENDER_MIN_CONF else "Unknown"
        age_text = a["label"] if a["confidence"] >= AGE_MIN_CONF else "Unknown"

    # ---- printed summary
    rows = [{"Question": "Real or fake?", "Answer": rf["label"].upper(), "Confidence": f"{rf['confidence']:.1%}"}]
    if "gender" in results:
        rows.append({"Question": "Gender", "Answer": gender_text, "Confidence": f"{results['gender']['confidence']:.1%}"})
        rows.append({"Question": "Age group", "Answer": age_text, "Confidence": f"{results['age']['confidence']:.1%}"})
    else:
        rows.append({"Question": "Gender", "Answer": gender_text, "Confidence": "rule (fake image)"})
        rows.append({"Question": "Age group", "Answer": age_text, "Confidence": "rule (fake image)"})
    print(pd.DataFrame(rows).to_string(index=False))

    # ---- figure: image + Grad-CAM for each model that ran
    panels = [
        (rf["original"], "Image (as seen by the real/fake model)"),
        (rf["gradcam"], f"Real/fake Grad-CAM: {rf['label'].upper()} ({rf['confidence']:.0%})"),
    ]
    if "gender" in results:
        g, a = results["gender"], results["age"]
        panels.append((g["gradcam"], f"Gender Grad-CAM: {g['label']} ({g['confidence']:.0%})"))
        panels.append((a["gradcam"], f"Age Grad-CAM: {a['label']} ({a['confidence']:.0%})"))

    fig = make_subplots(rows=1, cols=len(panels), subplot_titles=[p[1] for p in panels],
                        horizontal_spacing=0.015)
    for k, (img, _) in enumerate(panels, start=1):
        fig.add_trace(go.Image(z=img), row=1, col=k)

    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    fig.update_annotations(font_size=11)
    fig.update_layout(
        title=f"{rf['label'].upper()}  |  Gender: {gender_text}  |  Age group: {age_text}",
        width=300 * len(panels) + 40,
        height=430,
    )
    return fig, results

### Run it on one picture:

Set `IMAGE_PATH` in the settings cell to the picture you want, then run this cell. If `IMAGE_PATH` is `None`, a random real/fake test image is used.

In [24]:
if IMAGE_PATH is not None:
    image_path = Path(IMAGE_PATH)
else:
    image_path = random.choice(list(TEST_DIRS["real_fake"].rglob("*.jpg")))

print("Image:", image_path)
known = true_labels(image_path)
if known:
    print("Known labels in the dataset ->", known)
print()

picture = Image.open(image_path).convert("RGB")
if min(picture.size) < 256:
    print("Warning: this image is smaller than 256 px. The models were trained on larger images.\n")

fig, results = full_prediction(picture)
fig.show()

if SAVE_FIG:
    joblib.dump(fig, MODELS_DIR / "grad_cam_upload_fig.pkl")
    print("Figure saved as grad_cam_upload_fig.pkl")

Image: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\split_real_fake_norm\test\real\3011.jpg
Known labels in the dataset -> real | gender: female | age group: 26-35

     Question Answer Confidence
Real or fake?   REAL     100.0%
       Gender Female      74.7%
    Age group  26-35      56.9%


### Probability of every class (for the picture above):

In [26]:
for task, res in results.items():
    shown = {k: f"{v:.1%}" for k, v in sorted(res["probs"].items(), key=lambda kv: -kv[1])}
    print(f"{task:10s} {shown}")

real_fake  {'real': '100.0%', 'fake': '0.0%'}
gender     {'female': '74.7%', 'male': '25.3%'}
age        {'26-35': '56.9%', '18-25': '25.5%', '36-50': '14.9%', 'under_18': '1.9%', '50+': '0.9%', 'not_applicable': '0.0%'}


### Optional: upload a picture with a button

Run this cell, click **Upload**, choose a picture, then click **Analyse uploaded image**.
It needs the package `ipywidgets` (`pip install ipywidgets`). If it does not work in your editor, use `IMAGE_PATH` above instead.

In [27]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    uploader = widgets.FileUpload(accept="image/*", multiple=False)
    button = widgets.Button(description="Analyse uploaded image")
    output = widgets.Output()

    def on_click(_):
        with output:
            clear_output()
            if not uploader.value:
                print("Upload a picture first.")
                return
            item = uploader.value[0] if isinstance(uploader.value, (list, tuple)) else list(uploader.value.values())[0]
            img = Image.open(io.BytesIO(bytes(item["content"]))).convert("RGB")
            fig_up, _ = full_prediction(img)
            fig_up.show()

    button.on_click(on_click)
    display(uploader, button, output)

except ImportError:
    print("ipywidgets is not installed. Run: pip install ipywidgets  (or use IMAGE_PATH in the settings cell)")

ipywidgets is not installed. Run: pip install ipywidgets  (or use IMAGE_PATH in the settings cell)


# Grad-CAM on test images of each task

These cells show random **test** images (the models never trained on them) with the true class, the predicted class and the Grad-CAM heatmap, one row per class.

In [28]:
def show_task_examples(task, per_class=2, save_name=None):
    cfg = LOADED[task]
    classes_shown = [c for c in cfg["classes"] if c not in cfg["exclude"]]

    # random test images of each class
    picks = {}
    for cls in classes_shown:
        files = sorted((TEST_DIRS[task] / cls).glob("*.jpg"))
        picks[cls] = random.sample(files, min(per_class, len(files)))

    # analyse them
    analysed = {}
    for cls, files in picks.items():
        analysed[cls] = [analyse(task, Image.open(f).convert("RGB")) for f in files]

    # titles in row order: [image, Grad-CAM] for each example
    titles = []
    for cls in classes_shown:
        for k in range(per_class):
            if k < len(analysed[cls]):
                r = analysed[cls][k]
                status = "correct" if r["label"] == cls else "WRONG"
                titles.append(f"True: {cls} | Pred: {r['label']} ({r['confidence']:.0%}) {status}")
                titles.append(f"Grad-CAM for {r['label']}")
            else:
                titles += ["", ""]

    fig = make_subplots(rows=len(classes_shown), cols=2 * per_class, subplot_titles=titles,
                        vertical_spacing=0.08, horizontal_spacing=0.015)

    for row_number, cls in enumerate(classes_shown, start=1):
        for k, r in enumerate(analysed[cls]):
            fig.add_trace(go.Image(z=r["original"]), row=row_number, col=2 * k + 1)
            fig.add_trace(go.Image(z=r["gradcam"]), row=row_number, col=2 * k + 2)

    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    fig.update_annotations(font_size=10)
    fig.update_layout(
        title=f"{task} ({cfg['arch']}) - Grad-CAM on random test images, one row per true class",
        width=290 * 2 * per_class + 40,
        height=310 * len(classes_shown) + 90,
    )
    fig.show()

    if save_name and SAVE_FIG:
        joblib.dump(fig, MODELS_DIR / save_name)
        print("saved", save_name)
    return fig

### Real / fake:

In [29]:
fig_real_fake = show_task_examples("real_fake", per_class=2, save_name="grad_cam_real_fake_fig.pkl")

### Gender:

In [22]:
fig_gender = show_task_examples("gender", per_class=2, save_name="grad_cam_gender_fig.pkl")

### Age group:

In [30]:
fig_age = show_task_examples("age", per_class=2, save_name="grad_cam_age_fig.pkl")

# Notes for the report

- **Grad-CAM is an explanation, not a proof.** It shows which image regions influenced the answer. A heatmap on the face does not mean the answer is right, and a heatmap on the background is a warning sign.
- **Real / fake:** the models score about 100% on this dataset, but real and fake images come from different sources (real photos vs. 256 x 256 generated images). A model can be right because of how the images look overall, not because of a real deepfake artifact. If the heatmap lands on the background or the borders, this is the likely reason. Test with images from other sources before trusting the result outside this dataset.
- **Gender:** trained on real images only (female / male). Fake images are shown as *Unknown* by rule.
- **Age group:** this model was trained on the original `split_age` folder, in which most fake images carry the label `not_applicable`. The accuracy reported in its training notebook (about 78%) is therefore mostly the easy fake-vs-real decision; on the real images alone it is lower (about 58%, estimated from the reported numbers). Neighbouring age groups (26-35 and 36-50) are often confused. For a cleaner age model, retrain on real images only.
- **Confidence limits** (`GENDER_MIN_CONF`, `AGE_MIN_CONF`) turn uncertain answers into *Unknown*. Change them in the settings cell.